In [1]:
import pandas as pd
import re

def process_cases_to_csv():
    # 1. Load your master law dictionary (the CSV we just built)
    try:
        law_dict_df = pd.read_csv('law_table.csv')
        # Create a fast lookup dictionary: { '10.1.1': {crime_name_mn: 'Хүнийг алах'...} }
        law_lookup = law_dict_df.set_index('article_code').to_dict('index')
    except FileNotFoundError:
        print("Error: Make sure 'law_table.csv' is in the same folder as this script.")
        return

    # 2. Sample scraped data (In reality, you'd load this from your Playwright scraper)
    scraped_cases = [
        {"case_id": "Case_A_001", "decision_text": "Шүүгдэгч нь Эрүүгийн хуулийн 17.1.1-д заасан хулгайлах гэмт хэрэг үйлдсэн..."},
        {"case_id": "Case_B_002", "decision_text": "Эрүүгийн хуулийн тусгай ангийн 10.6.1, мөн хуулийн ерөнхий ангийн 6.7.1-ийг баримтлан..."},
        {"case_id": "Case_C_003", "decision_text": "Шүүхээс 25.3.1 болон 25.5.1 дүгээр зүйлд зааснаар ял оноов."}
    ]

    results = []

    # Regex pattern to catch Mongolian Criminal Code article numbers (e.g., 10.1.1, 17.2.1)
    # It looks for 1 to 2 digits, a dot, 1 to 2 digits, a dot, and 1 to 2 digits.
    article_pattern = re.compile(r'\b(\d{1,2}\.\d{1,2}\.\d{1,2})\b')

    print("Scanning case texts for legal codes...")
    
    for case in scraped_cases:
        text = case['decision_text']
        case_id = case['case_id']

        # Find all matching codes in the text
        found_codes = article_pattern.findall(text)

        # Deduplicate codes for this specific case so we don't count the same charge twice
        unique_codes = list(set(found_codes))

        for code in unique_codes:
            # Check if we have this code in our master dictionary
            if code in law_lookup:
                crime_info = law_lookup[code]
                results.append({
                    "case_id": case_id,
                    "article_code": code,
                    "chapter": crime_info['chapter'],
                    "part": crime_info['part'],
                    "crime_name_mn": crime_info['crime_name_mn'],
                    "crime_name_en": crime_info['crime_name_en']
                })
            else:
                # Catch codes that aren't in your dictionary yet so you know what to add later!
                results.append({
                    "case_id": case_id,
                    "article_code": code,
                    "chapter": "Unknown",
                    "part": "Unknown",
                    "crime_name_mn": "Needs Translation",
                    "crime_name_en": "Needs Translation"
                })

    # 3. Convert results to a DataFrame and export to CSV
    results_df = pd.DataFrame(results)
    
    # CRITICAL: encoding='utf-8-sig' ensures Excel reads Mongolian Cyrillic characters correctly
    output_filename = 'mapped_case_data.csv'
    results_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

    print(f"Success! Data successfully mapped and saved to {output_filename}")
    print("\nPreview of your new CSV data:")
    print(results_df.head(10))

# Run the function
if __name__ == "__main__":
    process_cases_to_csv()

Scanning case texts for legal codes...
Success! Data successfully mapped and saved to mapped_case_data.csv

Preview of your new CSV data:
      case_id article_code  chapter     part  \
0  Case_A_001       17.1.1  Unknown  Unknown   
1  Case_B_002       10.6.1       10  special   
2  Case_B_002        6.7.1        6  general   
3  Case_C_003       25.3.1       25  special   
4  Case_C_003       25.5.1       25  special   

                                       crime_name_mn  \
0                                  Needs Translation   
1  Хууль/гэрээгээр хүлээсэн үүргээ биелүүлээгүйн ...   
2                 Ялаас чөлөөлөх / хөнгөрүүлэх журам   
3                           Соёлын биет өвийг завших   
4  Соёлын биет өвийн худалдааг хууль бусаар явуул...   

                                       crime_name_en  
0                                  Needs Translation  
1  Death caused by failure to fulfil a legal/cont...  
2  General-part rule on mitigating/reducing sente...  
3     Misappropr